In [1]:
from lexico import *
from sintactico_ast_ext import *

In [2]:
# Ejemplo de uso
codigo_fuente = """
int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}
"""

In [3]:
# Anâlisis léxico 
tokens = identificar_tokens(codigo_fuente)
tokens

[('KEYWORD', 'int'),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'a'),
 ('DELIMITER', ','),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '+'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'main'),
 ('DELIMITER', '('),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'resultado'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('NUMBER', '3'),
 ('DELIMITER', ','),
 ('NUMBER', '5'),
 ('DELIMITER', ')'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('NUMBER', '0'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}')]

In [4]:
print('Tokens encontrados')
for tipo, valor in tokens:
    print(f'{tipo}: {valor}')

Tokens encontrados
KEYWORD: int
IDENTIFIER: suma
DELIMITER: (
KEYWORD: int
IDENTIFIER: a
DELIMITER: ,
KEYWORD: int
IDENTIFIER: b
DELIMITER: )
DELIMITER: {
KEYWORD: int
IDENTIFIER: c
OPERATOR: =
IDENTIFIER: a
OPERATOR: +
IDENTIFIER: b
DELIMITER: ;
KEYWORD: return
IDENTIFIER: c
DELIMITER: ;
DELIMITER: }
KEYWORD: int
IDENTIFIER: main
DELIMITER: (
DELIMITER: )
DELIMITER: {
KEYWORD: int
IDENTIFIER: resultado
OPERATOR: =
IDENTIFIER: suma
DELIMITER: (
NUMBER: 3
DELIMITER: ,
NUMBER: 5
DELIMITER: )
DELIMITER: ;
KEYWORD: return
NUMBER: 0
DELIMITER: ;
DELIMITER: }


In [5]:
try:
    print('\nIniciando analisis sintactico...')
    parser = Parser(tokens)
    arbol_ast = parser.parsear()
    print('Analisis sintactico completo sin errores!')
except SyntaxError as e:
    print(e)


Iniciando analisis sintactico...
Analisis sintactico completo sin errores!


In [6]:
import json 

In [7]:
def imprimir_ast(nodo):
    if isinstance(nodo, NodoPrograma):
        return {
            'programa': 'Noname',
            'funciones': [imprimir_ast(f) for f in nodo.funciones],
            'main': imprimir_ast(nodo.main)
        }
    elif isinstance(nodo, NodoFuncion):
        return {
            'nombre': nodo.nombre[1],
            'parametros': [imprimir_ast(p) for p in nodo.parametros],
            'cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]
        }
    elif isinstance(nodo, NodoParametro):          # corregido: sin 's'
        return {
            'Parametro': nodo.nombre[1],
            'Tipo': nodo.tipo[1]
        }
    elif isinstance(nodo, NodoAsignacion):
        return {
            'tipo': 'asignacion',
            'variable': nodo.nombre[1],
            'expresion': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            'op': nodo.operador[1],
            'izq': imprimir_ast(nodo.izquierda),
            'der': imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoRetorno):
        return {
            'tipo': 'Return',
            'valor': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoIdentificador):
        return nodo.nombre[1]
    elif isinstance(nodo, NodoNumero):
        return int(nodo.valor[1]) if str(nodo.valor[1]).isdigit() else nodo.valor[1]
    elif isinstance(nodo, NodoCadena):
        return nodo.valor[1]
    elif isinstance(nodo, NodoPrint):              # corregido: NodoPrint
        return {
            'tipo': 'print',
            'argumentos': [imprimir_ast(a) for a in nodo.argumentos]
        }
    elif isinstance(nodo, NodoPrintf):             # corregido: NodoPrintf
        return {
            'tipo': 'printf',
            'argumentos': [imprimir_ast(a) for a in nodo.argumentos]
        }
    elif isinstance(nodo, NodoLlamadaFuncion):
        return {
            'tipo': 'llamada',
            'funcion': nodo.nombre_funcion,
            'argumentos': [imprimir_ast(a) for a in nodo.argumentos]
        }
    else:
        return {}
        

In [8]:
print(json.dumps(imprimir_ast(arbol_ast), indent=1))

{
 "programa": "Noname",
 "funciones": [
  {
   "nombre": "suma",
   "parametros": [
    {
     "Parametro": "a",
     "Tipo": "int"
    },
    {
     "Parametro": "b",
     "Tipo": "int"
    }
   ],
   "cuerpo": [
    {
     "tipo": "asignacion",
     "variable": "c",
     "expresion": {
      "op": "+",
      "izq": "a",
      "der": "b"
     }
    },
    {
     "tipo": "Return",
     "valor": "c"
    }
   ]
  }
 ],
 "main": {
  "nombre": "main",
  "parametros": [],
  "cuerpo": [
   {
    "tipo": "asignacion",
    "variable": "resultado",
    "expresion": {
     "tipo": "llamada",
     "funcion": "suma",
     "argumentos": [
      3,
      5
     ]
    }
   },
   {
    "tipo": "Return",
    "valor": 0
   }
  ]
 }
}


In [9]:
nodoExp = NodoOperacion(NodoNumero(('NUM', 5)), ('OPERATOR', '+'), NodoNumero(('NUM', 8)))
print(json.dumps(imprimir_ast(nodoExp), indent=1))

{
 "op": "+",
 "izq": 5,
 "der": 8
}


In [10]:
print(codigo_fuente)


int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}



In [11]:
print(arbol_ast.traducirPy())

def suma(a, b):
    c = a + b
    return c

def main():
    resultado = suma(3, 5)
    return 0


In [12]:
print(arbol_ast.traducirRuby())

def suma(a, b)
  c = a + b
  return c
end

def main()
  resultado = suma(3, 5)
  return 0
end


In [13]:

# Ejemplo de uso
codigo_fuente_main = """
int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}
"""

In [14]:
# Anâlisis léxico 
tokens_new = identificar_tokens(codigo_fuente_main)
tokens_new

[('KEYWORD', 'int'),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'a'),
 ('DELIMITER', ','),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '+'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'main'),
 ('DELIMITER', '('),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'resultado'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('NUMBER', '3'),
 ('DELIMITER', ','),
 ('NUMBER', '5'),
 ('DELIMITER', ')'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('NUMBER', '0'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}')]

In [15]:
 # Analisis sintactico
try:
    print('\nIniciando análisis sintáctico...')
    parser_main = Parser(tokens_new)
    arbol_ast_main = parser_main.parsear()
    print('Analisis sintactico completo sin errores!')
except SyntaxError as e:
    print(e)
    


Iniciando análisis sintáctico...
Analisis sintactico completo sin errores!


In [16]:
print(json.dumps(imprimir_ast(arbol_ast_main), indent=1))

{
 "programa": "Noname",
 "funciones": [
  {
   "nombre": "suma",
   "parametros": [
    {
     "Parametro": "a",
     "Tipo": "int"
    },
    {
     "Parametro": "b",
     "Tipo": "int"
    }
   ],
   "cuerpo": [
    {
     "tipo": "asignacion",
     "variable": "c",
     "expresion": {
      "op": "+",
      "izq": "a",
      "der": "b"
     }
    },
    {
     "tipo": "Return",
     "valor": "c"
    }
   ]
  }
 ],
 "main": {
  "nombre": "main",
  "parametros": [],
  "cuerpo": [
   {
    "tipo": "asignacion",
    "variable": "resultado",
    "expresion": {
     "tipo": "llamada",
     "funcion": "suma",
     "argumentos": [
      3,
      5
     ]
    }
   },
   {
    "tipo": "Return",
    "valor": 0
   }
  ]
 }
}


In [17]:
nodo_programa = NodoPrograma(
    funciones={arbol_ast},
    main = arbol_ast_main
)

In [18]:
codigo = arbol_ast.generarCodigo()

In [19]:
print(codigo)

section .bss
    c:    resd 1
    a:    resd 1
    b:    resd 1
    resultado:    resd 1
section .text
global _start
suma:

    pop    eax
    mov [a], eax
    pop    eax
    mov [b], eax
    mov eax, [a]
    push    eax

    mov eax, [b]
    mov    ebx, eax
    pop    eax
    add    eax, ebx
    mov [c], eax

    mov eax, [c]
    ret

_start:
main:

    mov eax, 5
    push eax  ; Pasar argumento a la pila

    mov eax, 3
    push eax  ; Pasar argumento a la pila
    call suma  ; Llamar a la funcion suma
    add esp, 8  ; Limpiar pila de argumentos
    mov [resultado], eax

    mov eax, 0
    ret

    mov eax, 1  ; syscall exit
    xor ebx, ebx  ; codigo de salida 0
    int 0x80


In [20]:
import semantico_ext

In [21]:
analizador_semantico = semantico_ext.AnalizadorSemantico()
analisis = analizador_semantico.analizar(arbol_ast)

int
int


In [22]:
analizador_semantico.tabla_simbolos.funciones

{'suma': ('int', [('a', 'int'), ('b', 'int')]), 'main': ('int', [])}

In [23]:
analizador_semantico.tabla_simbolos.variables 

{'a': 'int', 'b': 'int', 'c': 'int', 'resultado': 'int'}

In [24]:
nodo_exp = NodoOperacion(NodoNumero(('NUMBER','1')),('OPERATOR','*'), NodoIdentificador(('IDENTIFIER','algo')))
print(json.dumps(imprimir_ast(nodo_exp), indent=1))

{
 "op": "*",
 "izq": 1,
 "der": "algo"
}


In [25]:
exp_op = nodo_exp.optimizar()
print(json.dumps(imprimir_ast(exp_op), indent=1))

"algo"


In [26]:
import subprocess

In [27]:

def compilar(programa):
    # Generar el codigo en ensamblador
    codigo_asm = programa.generarCodigo()
    print(codigo_asm)
    text_file = open("ejasmcompiladores.asm", "w")
    text_file.write(codigo_asm)
    text_file.close()
    subprocess.run(["nasm", "-f", "elf", "ejasmcompiladores.asm"])
    subprocess.run(["ld", "-m", "elf_i386", "ejasmcompiladores.o", "-o", "ejasmcompiladores"])
    

In [28]:
compilar(arbol_ast)

section .bss
    c:    resd 1
    a:    resd 1
    b:    resd 1
    resultado:    resd 1
section .text
global _start
suma:

    pop    eax
    mov [a], eax
    pop    eax
    mov [b], eax
    mov eax, [a]
    push    eax

    mov eax, [b]
    mov    ebx, eax
    pop    eax
    add    eax, ebx
    mov [c], eax

    mov eax, [c]
    ret

_start:
main:

    mov eax, 5
    push eax  ; Pasar argumento a la pila

    mov eax, 3
    push eax  ; Pasar argumento a la pila
    call suma  ; Llamar a la funcion suma
    add esp, 8  ; Limpiar pila de argumentos
    mov [resultado], eax

    mov eax, 0
    ret

    mov eax, 1  ; syscall exit
    xor ebx, ebx  ; codigo de salida 0
    int 0x80


ld: file cannot be open()ed, errno=2 path=elf_i386


In [29]:
print(codigo_fuente)



int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}



In [ ]:
graph TD 
    subgraph main
        M1(Inicio main) --> M2[resultado = suma 3,5]
        M2 --> M3[return 0]
        M3 --> M4(Fin main)
    end 

    subgraph suma 
        S1(Inicio_suma) --> S2[c = a + b]
        S2 --> S3[return c]
        S3 --> S4(Fin suma)
    end 




In [ ]:
graph TD 
    Start([Inicio]) --> Init[n = 5]
    Init --> Condition{¿n > 0?)

    Condition -- Si --> Show[/Mostrar n/]
    show --> Dec[n = n - 1 ]
    Dec --> Condition

    Condition --> No --> End([Fin])